<a href="https://colab.research.google.com/github/customerdelightgroup/customerdelight/blob/GroupCode/Building_a_Customer_Delight_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df = pd.read_csv("yelp_labelled.txt", sep="\t", header=None, names=["sentence", "label"])

# Show first rows
print(df.head())

# Check size
print("\nShape:", df.shape)

# Check labels
print("\nLabel counts:")
print(df["label"].value_counts())

                                            sentence  label
0                           Wow... Loved this place.      1
1                                 Crust is not good.      0
2          Not tasty and the texture was just nasty.      0
3  Stopped by during the late May bank holiday of...      1
4  The selection on the menu was great and so wer...      1

Shape: (1000, 2)

Label counts:
label
1    500
0    500
Name: count, dtype: int64


Clean the text This step improves model accuracy

In [ ]:
import re

def clean_text(text):
    text = text.lower()  # lowercase
    text = re.sub(r"[^a-z\s]", "", text)  # remove punctuation & numbers
    return text

# Apply cleaning
df["clean_sentence"] = df["sentence"].apply(clean_text)

# Show results
print(df[["sentence", "clean_sentence"]].head())

                                            sentence  \
0                           Wow... Loved this place.   
1                                 Crust is not good.   
2          Not tasty and the texture was just nasty.   
3  Stopped by during the late May bank holiday of...   
4  The selection on the menu was great and so wer...   

                                      clean_sentence  
0                               wow loved this place  
1                                  crust is not good  
2           not tasty and the texture was just nasty  
3  stopped by during the late may bank holiday of...  
4  the selection on the menu was great and so wer...  


We converted all text to lowercase and removed punctuation, symbols, and numbers to make the data clean and consistent for analysis.

In [ ]:
import re

def clean_text(text):
    text = text.lower()  # lowercase
    text = re.sub(r"[^a-z\s]", "", text)  # remove punctuation & numbers
    return text

# Apply cleaning
df["clean_sentence"] = df["sentence"].apply(clean_text)

# Show results
print(df[["sentence", "clean_sentence"]].head())

                                            sentence  \
0                           Wow... Loved this place.   
1                                 Crust is not good.   
2          Not tasty and the texture was just nasty.   
3  Stopped by during the late May bank holiday of...   
4  The selection on the menu was great and so wer...   

                                      clean_sentence  
0                               wow loved this place  
1                                  crust is not good  
2           not tasty and the texture was just nasty  
3  stopped by during the late may bank holiday of...  
4  the selection on the menu was great and so wer...  


In this step, we tokenized the text into individual words and removed common stopwords such as "the", "is", and "and". This helps focus on important words that contribute to sentiment analysis.

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def remove_stopwords(text):
    words = text.split()  # tokenize
    filtered_words = [word for word in words if word not in ENGLISH_STOP_WORDS]
    return " ".join(filtered_words)

# Apply stopword removal
df["final_text"] = df["clean_sentence"].apply(remove_stopwords)

# Show results
print(df[["clean_sentence", "final_text"]].head())

                                      clean_sentence  \
0                               wow loved this place   
1                                  crust is not good   
2           not tasty and the texture was just nasty   
3  stopped by during the late may bank holiday of...   
4  the selection on the menu was great and so wer...   

                                          final_text  
0                                    wow loved place  
1                                         crust good  
2                           tasty texture just nasty  
3  stopped late bank holiday rick steve recommend...  
4                        selection menu great prices  


We split the dataset into training and testing sets. The training set is used to train the model, while the testing set is used to check how well the model performs on unseen data.

In [ ]:
from sklearn.model_selection import train_test_split

X = df["final_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set size:", len(X_train))
print("Testing set size:", len(X_test))

Training set size: 800
Testing set size: 200


In this step, we used TF-IDF to convert the text into numerical features that a machine learning model can understand. TF-IDF gives higher importance to words that are more meaningful in a sentence and lower importance to very common words.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("X_train_tfidf shape:", X_train_tfidf.shape)
print("X_test_tfidf shape:", X_test_tfidf.shape)

X_train_tfidf shape: (800, 1590)
X_test_tfidf shape: (200, 1590)


In this step, we trained a Multinomial Naive Bayes model using the TF-IDF features. This model is commonly used for text classification because it works well with word-based data.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In this step, we evaluated the Naive Bayes model using accuracy, classification report, and confusion matrix. These metrics help measure how well the model classifies positive and negative reviews.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

y_pred_nb = nb_model.predict(X_test_tfidf)

print("Predictions completed.")

Predictions completed.


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_nb))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_nb))

Accuracy: 0.785

Classification Report:

              precision    recall  f1-score   support

           0       0.77      0.79      0.78        96
           1       0.80      0.78      0.79       104

    accuracy                           0.79       200
   macro avg       0.78      0.79      0.78       200
weighted avg       0.79      0.79      0.79       200


Confusion Matrix:

[[76 20]
 [23 81]]


In [ ]:
print(type(y_pred_nb))

<class 'numpy.ndarray'>


In this step, we trained a Logistic Regression model using the TF-IDF features. We used this model to compare its performance with the Naive Bayes model.

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)

y_pred_lr = lr_model.predict(X_test_tfidf)

print("Logistic Regression predictions completed.")

Logistic Regression predictions completed.


In this step, we evaluated the Logistic Regression model using accuracy, classification report, and confusion matrix. This helped us compare its performance with the Naive Bayes model.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_lr))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_lr))

Accuracy: 0.785

Classification Report:

              precision    recall  f1-score   support

           0       0.74      0.85      0.79        96
           1       0.84      0.72      0.78       104

    accuracy                           0.79       200
   macro avg       0.79      0.79      0.78       200
weighted avg       0.79      0.79      0.78       200


Confusion Matrix:

[[82 14]
 [29 75]]


In this step, we compared the accuracy of the Naive Bayes model and the Logistic Regression model. This helped us identify which model performed better on the test data.

In [ ]:
nb_accuracy = accuracy_score(y_test, y_pred_nb)
lr_accuracy = accuracy_score(y_test, y_pred_lr)

print("Naive Bayes Accuracy:", nb_accuracy)
print("Logistic Regression Accuracy:", lr_accuracy)

if lr_accuracy > nb_accuracy:
    print("Logistic Regression performed better.")
elif nb_accuracy > lr_accuracy:
    print("Naive Bayes performed better.")
else:
    print("Both models performed the same.")

Naive Bayes Accuracy: 0.785
Logistic Regression Accuracy: 0.785
Both models performed the same.


In this step, we identified the most important words contributing to positive and negative sentiment. This helps understand the key factors influencing customer satisfaction and dissatisfaction.